# Malaysia Essential Food Price Dispersion Monitor

**Policy question:** Where are selected essential-food prices unusually dispersed across Malaysian states and premise types, and when do local price movements merit closer investigation?

PriceCatcher is used as **surveillance evidence**, not as an alternative CPI.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data" / "processed" / "state_item_month_diagnostics.csv"
df = pd.read_csv(DATA, parse_dates=["month"])
df.head()


## 1. Coverage first

Before interpreting prices, inspect the number of observations and distinct premises. A high price based on thin coverage should not trigger the same response as a pattern observed across many premises.

In [ ]:
df[["n_observations", "n_premises"]].describe()


## 2. Latest-month price gaps

The state-to-national gap compares the state median with the national median for the **same item and month**.

In [ ]:
latest = df["month"].max()
latest_df = df[df["month"].eq(latest)].copy()

latest_df.sort_values(
    "state_to_national_gap",
    ascending=False
)[["state","item","unit","n_observations","n_premises",
   "price_median","national_price_median","state_to_national_gap"]].head(20)


## 3. Price dispersion

A wide IQR relative to the median can indicate heterogeneous local prices. It may also reflect premise mix or product-specification problems, so the result is a prompt for diagnostic review.

In [ ]:
latest_df.sort_values(
    "dispersion_iqr_over_median",
    ascending=False
)[["state","item","unit","dispersion_iqr_over_median",
   "n_observations","n_premises"]].head(20)


## 4. Robust time-series anomalies

The robust z-score compares a state-item monthly median with its own series using median absolute deviation. Extreme scores are unusual observations, not proof of a shock.

In [ ]:
df.reindex(
    df["robust_z"].abs().sort_values(ascending=False).index
)[["month","state","item","price_median","robust_z","review_flag"]].head(20)


## 5. Policy interpretation

A useful analyst workflow is:

1. confirm coverage;
2. confirm product unit/specification;
3. inspect premise and district mix;
4. compare with the item's own recent history;
5. check whether the pattern is isolated or persistent;
6. use CPI for inflation context;
7. escalate only after the evidence supports a specific policy question.

This preserves the distinction between **screening** and **finding**.